
# NLF Gait Demo (Video → Frames → NLF → 3D Joints)

This notebook:

1. Installs dependencies (PyTorch, torchvision, OpenCV, smplx, matplotlib).
2. Downloads the NLF TorchScript model.
3. Lets you upload a running video (e.g. `run.mp4`).
4. Extracts frames from the video.
5. Runs NLF on each frame to get **SMPL-style 3D joints and vertices**.
6. Visualizes the 3D skeleton sequence as an animation.

> **Note:** NLF already uses a SMPL parametric head internally and outputs `joints3d` and `vertices3d` in SMPL space, so no external SMPL model file is needed just to visualize the body or analyze gait.


In [ ]:

# Install dependencies (run once, especially in Colab)
!pip install -q torch torchvision smplx opencv-python matplotlib imageio


In [ ]:

import os
import glob
import numpy as np
import torch
import torchvision
import cv2
import imageio
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

# Download NLF TorchScript model
os.makedirs("models", exist_ok=True)

# Large model (~300MB). This is the same file as in the official NLF demo.
!wget -q -O models/nlf_l_multi.torchscript https://bit.ly/nlf_l_pt

model_path = "models/nlf_l_multi.torchscript"
assert os.path.exists(model_path), "Model download failed"

model = torch.jit.load(model_path, map_location=device).eval()
print("Loaded NLF model from", model_path)



## Upload a running video

Upload a video file (e.g. `run.mp4`) that shows a single runner.
The next cell will save it and extract frames from it.


In [ ]:

from google.colab import files

uploaded = files.upload()  # choose your running video file
assert len(uploaded) > 0, "Please upload at least one video file."

video_filename = list(uploaded.keys())[0]
print("Using video:", video_filename)


In [ ]:

# Extract frames from the uploaded video

video_path = video_filename
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video FPS: {fps:.2f}, total frames: {frame_count}")

frame_idx = 0
saved = 0
frame_stride = 1  # change to >1 to subsample frames

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_stride == 0:
        out_path = os.path.join(frames_dir, f"{frame_idx:05d}.jpg")
        cv2.imwrite(out_path, frame)
        saved += 1
    frame_idx += 1

cap.release()
print("Saved", saved, "frames to", frames_dir)



## Run NLF on frames to get 3D joints & vertices

For each frame, we:

1. Load the image as a PyTorch tensor.
2. Run `model.detect_smpl_batched` (parametric SMPL head).
3. Store:
   * `joints3d` – SMPL 3D joints (24 × 3)
   * `vertices3d` – SMPL mesh vertices (6890 × 3)


In [ ]:

def load_image_for_nlf(path: str, device: torch.device):
    img = torchvision.io.read_image(path)  # [3, H, W], uint8
    img = img.to(device)
    return img

frame_paths = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
assert frame_paths, "No frames found. Check that extraction worked."

all_joints3d = []
all_vertices3d = []
used_frame_paths = []

with torch.inference_mode(), torch.device(device):
    for fp in frame_paths:
        img = load_image_for_nlf(fp, device)
        batch = img.unsqueeze(0)  # [1, 3, H, W]
        pred = model.detect_smpl_batched(batch)

        joints3d = pred["joints3d"][0].detach().cpu().numpy()      # (24, 3)
        vertices3d = pred["vertices3d"][0].detach().cpu().numpy()  # (6890, 3)

        all_joints3d.append(joints3d)
        all_vertices3d.append(vertices3d)
        used_frame_paths.append(fp)

all_joints3d = np.stack(all_joints3d, axis=0)       # (T, 24, 3)
all_vertices3d = np.stack(all_vertices3d, axis=0)   # (T, V, 3)

print("joints3d shape:", all_joints3d.shape)
print("vertices3d shape:", all_vertices3d.shape)
print("Frames processed:", len(used_frame_paths))



## Optional: 2D overlay check

Overlay 2D joints on top of one frame to verify that the model is tracking the runner correctly.


In [ ]:

idx = 0  # which frame to visualize
test_frame_path = used_frame_paths[idx]
print("Visualizing frame:", test_frame_path)

bgr = cv2.imread(test_frame_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

with torch.inference_mode(), torch.device(device):
    img = torchvision.io.read_image(test_frame_path).to(device)
    batch = img.unsqueeze(0)
    pred_vis = model.detect_smpl_batched(batch)

joints2d = pred_vis["joints2d"][0].detach().cpu().numpy()  # (24, 2)

plt.figure(figsize=(4, 6))
plt.imshow(rgb)
plt.scatter(joints2d[:, 0], joints2d[:, 1], s=20, c="red")
for j, (x, y) in enumerate(joints2d):
    plt.text(x, y, str(j), color="yellow", fontsize=8)
plt.axis("off")
plt.title("2D joints (SMPL indices)")
plt.show()



## 3D Skeleton Animation (SMPL joints over time)

We now animate the 3D SMPL joints predicted by NLF over all frames.

We use the standard 24-joint SMPL kinematic tree to draw bones between joints.


In [ ]:

# Standard SMPL 24-joint parent relations (j -> parent_j)
parents = [
    -1,  # 0: pelvis
    0,   # 1: left_hip
    0,   # 2: right_hip
    0,   # 3: spine1
    1,   # 4: left_knee
    2,   # 5: right_knee
    3,   # 6: spine2
    4,   # 7: left_ankle
    5,   # 8: right_ankle
    6,   # 9: spine3 (chest)
    7,   # 10: left_foot
    8,   # 11: right_foot
    9,   # 12: neck
    12,  # 13: left_collar
    12,  # 14: right_collar
    12,  # 15: head
    13,  # 16: left_shoulder
    14,  # 17: right_shoulder
    16,  # 18: left_elbow
    17,  # 19: right_elbow
    18,  # 20: left_wrist
    19,  # 21: right_wrist
    20,  # 22: left_hand
    21,  # 23: right_hand
]

T = all_joints3d.shape[0]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")

all_pts = all_joints3d.reshape(-1, 3)
x_min, y_min, z_min = all_pts.min(axis=0)
x_max, y_max, z_max = all_pts.max(axis=0)

def update(frame_idx):
    ax.clear()
    J = all_joints3d[frame_idx]

    ax.scatter(J[:, 0], J[:, 1], J[:, 2], s=15)

    for j, p in enumerate(parents):
        if p < 0:
            continue
        xs = [J[j, 0], J[p, 0]]
        ys = [J[j, 1], J[p, 1]]
        zs = [J[j, 2], J[p, 2]]
        ax.plot(xs, ys, zs)

    ax.set_title(f"Frame {frame_idx+1}/{T}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_zlim(z_min, z_max)
    ax.view_init(elev=10, azim=-90)

ani = FuncAnimation(fig, update, frames=T, interval=100)
plt.show()



## Next steps: Gait Analysis

From `all_joints3d` (shape `(T, 24, 3)`), you can compute:

* Hip, knee, ankle angles over time.
* Stride length and cadence.
* Vertical oscillation of the pelvis.
* Symmetry between left and right leg.

Example indices:

```python
pelvis      = all_joints3d[:, 0]  # (T, 3)
left_hip    = all_joints3d[:, 1]
right_hip   = all_joints3d[:, 2]
left_knee   = all_joints3d[:, 4]
right_knee  = all_joints3d[:, 5]
left_ankle  = all_joints3d[:, 7]
right_ankle = all_joints3d[:, 8]
```

You can build any custom biomechanical metric from these trajectories.
